# Etapa 3 · Punto 5 — Sistema de recomendación

**Objetivo.** Ofrecer una forma sencilla de que una persona pida recomendaciones y las entienda. Escribe al menos una
preferencia inicial y recibe una lista ordenada, con una explicación de por qué se recomienda cada canción.

### Arquitectura

```
                       ┌────────────── Interfaz web (Gradio) · app/app.py ──────────────┐
  preferencia ───────► │ canciones · artistas · géneros · usuario CritiqueBrainz · texto │
                       └───────────────────────────────┬────────────────────────────────┘
                                                       ▼
            src/sistema_recomendacion.py  ── SistemaRecomendacion.recomendar(...)
              1. interpretar la preferencia → nodos semilla del grafo  (texto libre → índice vectorial, punto 2)
              2. candidatos: Personalized PageRank + escenas Louvain    (src/recomendador_grafo.py, punto 3)
              3. puntaje final y Top-K: grafo + valoración + afinidad   (src/ranking.py, punto 4)
              4. explicación: camino en el grafo, escena, valoración y fragmento de una reseña real (punto 2)
                                                       ▼
                                              Neo4j AuraDB (grafo + índice vectorial)
```

**Nada está precalculado.** En cada solicitud se ejecuta PPR sobre el grafo proyectado desde Neo4j, se calculan las
señales del ranking y se consulta el índice vectorial. La interfaz no contiene listas de recomendaciones.

### Cómo usarlo

| Dónde | Comando |
|---|---|
| Local (recomendado) | Desde la raíz del repo: `python entrega_3/app/app.py`. Se abre `http://127.0.0.1:7860` |
| Este notebook | La última sección lanza la misma interfaz |
| Google Colab | Subir la carpeta `entrega_3` (con `.env`), ejecutar este notebook y usar el enlace público que genera Gradio |

La primera carga tarda alrededor de 1–2 minutos, porque proyecta el grafo y lee las reseñas. Después, cada
recomendación tarda menos de un segundo. La primera consulta con texto libre carga además el modelo de embeddings.

## 0. Entorno

In [1]:
%pip -q install "neo4j>=5.26,<7" "networkx>=3.2" "pandas>=2" "scikit-learn>=1.3" "sentence-transformers>=3,<6" "gradio>=5,<6"

Note: you may need to restart the kernel to use updated packages.


In [2]:
import os, sys, time
from pathlib import Path
import pandas as pd
from IPython.display import HTML, Markdown, display

RAIZ = Path('..').resolve()                        # entrega_3/
sys.path[:0] = [str(RAIZ / 'src'), str(RAIZ / 'app')]
from sistema_recomendacion import SistemaRecomendacion
import app as interfaz

## 1. Inicializar el sistema

Se conecta a Neo4j con las credenciales de `entrega_3/.env` y prepara lo que necesitan los puntos 3 y 4:
- la proyección del grafo;
- las comunidades de Louvain;
- las reseñas con sus vectores;
- el modelo de valoración;
- los pesos de `data/ranking_config.json`, elegidos en el notebook 04.

In [3]:
t0 = time.time()
sistema = SistemaRecomendacion()
print(f'Sistema listo en {time.time() - t0:.0f}s | pesos del ranking: {sistema.ranking.pesos}')

Proyectando el grafo desde Neo4j y detectando comunidades (punto 3)...


Cargando reseñas, embeddings y valoraciones (puntos 2 y 4)...


Conexión perdida (ServiceUnavailable); reconectando 1/14...


Listo: 130,031 nodos, 133 escenas, 2,006 reseñas.
Sistema listo en 72s | pesos del ranking: {'G': 0.2, 'V': 0.1, 'A': 0.7, 'E': 0.0}


## 2. Ejemplos: lo que ve la persona usuaria

Cada ejemplo llama a `sistema.recomendar(...)`, la misma función que usa el botón de la interfaz, y muestra las
tarjetas tal como aparecen en la app. Cada tarjeta incluye:
- **¿Por qué?:** el motivo principal, sacado del camino más fuerte en el grafo;
- **Escena musical:** la comunidad de Louvain del artista;
- **Valoración de usuarios:** el promedio de estrellas, explícitas o estimadas desde el texto;
- **Fragmento de reseña:** el más afín a la preferencia, recuperado del índice vectorial;
- **Camino completo en el grafo:** desplegable.

In [4]:
def mostrar(**preferencia):
    r = sistema.recomendar(**preferencia)
    if 'error' in r:
        display(Markdown(f'⚠️ {r["error"]}'))
        return r
    display(Markdown('**Preferencia:** ' + '; '.join(b for _, b in r['entendido']) +
                     ('  \n**Escena:** ' + ', '.join(f'«{e}» ({w:.0%})' for e, w in r['escena']) if r['escena'] else '')))
    display(HTML(f'<style>{interfaz.CSS}</style>' + ''.join(interfaz.tarjeta(it, r['items'][0]['score']) for it in r['items'])))
    return r

### 2.1 Una canción

In [5]:
r = mostrar(canciones='Crazy Train', k=5)

**Preferencia:** «Crazy Train» (Ozzy Osbourne)  
**Escena:** «rock · hard rock · blues rock» (100%)

### 2.2 Un artista

In [6]:
r = mostrar(artistas='Bukka White', k=5)

**Preferencia:** Bukka White  
**Escena:** «blues» (100%)

### 2.3 Un usuario de CritiqueBrainz (sus reseñas definen el perfil)

In [7]:
usuario_demo = next(v for etiqueta, v in sistema.usuarios if etiqueta.startswith('Jaime Gill'))
r = mostrar(usuario=usuario_demo, k=5)

**Preferencia:** usuario Jaime Gill  
**Escena:** «escena de The Rascals / Weezer» (78%), «heavy metal · hip-hop · post-punk» (4%), «thrash metal · grunge · metalcore» (4%)

### 2.4 Texto libre en español

In [8]:
r = mostrar(texto='guitarras acústicas melancólicas, folk íntimo con voz suave', k=5)

W1118 15:41:19.988000 30453 site-packages/torch/distributed/elastic/multiprocessing/redirects.py:35] NOTE: Redirects are currently not supported in MacOs.


Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

**Preferencia:** reseñas parecidas sobre álbum «Tourist», álbum «Natural», álbum «Holes in the Wall»  
**Escena:** «escena de The Rascals / Weezer» (21%), «escena de Ojos De Brujo / Mariza» (20%), «heavy metal · hip-hop · post-punk» (20%)

### 2.5 Preferencia combinada, solo dentro de la misma escena

In [9]:
r = mostrar(artistas='Aerosmith', tags=['classic rock'], k=5, misma_escena=True)

**Preferencia:** Aerosmith; #classic rock  
**Escena:** «rock · hard rock · blues rock» (50%), «classic rock · christmas» (50%)

### 2.6 Una preferencia que no existe en el grafo

In [10]:
r = mostrar(artistas='Artista Inventado 123')

⚠️ No reconocí ninguna preferencia. No encontré: artista «Artista Inventado 123».

## 3. Las recomendaciones provienen del mecanismo de los puntos anteriores

Dos comprobaciones:

1. **Coherencia con el punto 4.** El notebook 04 guardó en Neo4j el Top-10 de cada usuario
   (`RECOMMENDED {method: 'ranking_final'}`). Si el sistema calcula en vivo la recomendación para esos mismos usuarios,
   debe obtener las mismas canciones en el mismo orden.
2. **Respuesta a la entrada.** Distintas preferencias producen listas distintas, y el tamaño de la lista sigue a `K`.

In [11]:
muestra = [v for _, v in sistema.usuarios][:: max(1, len(sistema.usuarios) // 25)][:25]
coinciden = 0
for u in muestra:
    vivo = [it['uri'] for it in sistema.recomendar(usuario=u, k=10)['items']]
    guardado = [f['t'] for f in sistema.run_cypher('''
        MATCH (:User {uri: $u})-[r:RECOMMENDED {method: 'ranking_final'}]->(t:Track)
        RETURN t.uri AS t ORDER BY r.rank''', u=u)]
    coinciden += vivo == guardado
print(f'Usuarios con el mismo Top-10 en vivo que en Neo4j (punto 4): {coinciden} de {len(muestra)}')

listas = {nombre: [it['cancion'] for it in sistema.recomendar(**pref)['items']]
          for nombre, pref in {'Crazy Train (K=5)': dict(canciones='Crazy Train', k=5),
                               'Crazy Train (K=8)': dict(canciones='Crazy Train', k=8),
                               'blues': dict(tags=['blues'], k=5)}.items()}
display(pd.DataFrame({k: pd.Series(v) for k, v in listas.items()}))

Conexión perdida (ServiceUnavailable); reconectando 1/14...


Neo4j no disponible (SessionExpired); reintento 1/14 en 5s...


Neo4j no disponible (ServiceUnavailable); reintento 2/14 en 5s...


Neo4j no disponible (ServiceUnavailable); reintento 3/14 en 5s...


Neo4j no disponible (ServiceUnavailable); reintento 4/14 en 5s...


Neo4j no disponible (ServiceUnavailable); reintento 5/14 en 5s...


... (3 outputs left out)

Neo4j no disponible (ServiceUnavailable); reintento 9/14 en 5s...


Neo4j no disponible (ServiceUnavailable); reintento 10/14 en 5s...


Usuarios con el mismo Top-10 en vivo que en Neo4j (punto 4): 25 de 25


,Crazy Train (K=5),Crazy Train (K=8),blues
0,The End Of The Line,The End Of The Line,The Things That I Used To Do
1,Angel,Angel,You Gotta Move
2,Only Lonely,Only Lonely,Bucket's Got A Hole In It
3,S.A.T.O.,S.A.T.O.,Gone Dead On You Blues
4,Can You Hear Them?,Can You Hear Them?,Meet Me in the City
5,NaN,I Am The Law,NaN
6,NaN,Iron Man,NaN
7,NaN,Rainbow In The Dark,NaN


## 4. Lanzar la interfaz

Al ejecutar la celda se abre la aplicación: en Jupyter o VS Code aparece en línea y en un enlace local; en Colab se
genera además un enlace público temporal. Para detenerla, ejecutar `demo.close()`.

<!-- Al ejecutar el notebook de forma automatizada (NB_SIN_INTERFAZ=1) esta celda se omite. -->

In [12]:
if os.environ.get('NB_SIN_INTERFAZ'):
    print('Interfaz omitida en la ejecución automatizada. Ejecuta esta celda (o `python entrega_3/app/app.py`) para abrirla.')
else:
    demo = interfaz.construir_interfaz(sistema)
    demo.launch(share='google.colab' in sys.modules, inline=True, prevent_thread_lock=True)

Interfaz omitida en la ejecución automatizada. Ejecuta esta celda (o `python entrega_3/app/app.py`) para abrirla.


## 5. Resumen

| Requisito | Cómo se cumple |
|---|---|
| Interactuar y solicitar recomendaciones | Interfaz web (Gradio), con formulario y botón. También funciona como función de Python: `sistema.recomendar(...)` |
| Al menos una preferencia inicial | Canciones, artistas, géneros/etiquetas, un usuario de CritiqueBrainz o una descripción libre (combinables). La interfaz confirma qué entendió y avisa lo que no encontró |
| Lista ordenada | Top-K configurable (3–20), con el puntaje final del punto 4 y diversidad de artistas |
| Proviene del mecanismo desarrollado | Cada solicitud ejecuta PPR + Louvain (punto 3), el ranking (punto 4) y el índice vectorial (punto 2). Coincide con el Top-10 que el punto 4 guardó en Neo4j |
| Explicación sencilla | Motivo en lenguaje natural a partir del camino del grafo, escena, valoración de usuarios, afinidad, fragmento de una reseña real con enlace y camino completo desplegable |